In [1]:
import os
from dotenv import load_dotenv
import requests

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent

load_dotenv()

@tool
def get_current_time(city: str) -> str:
    """Get the current local time for a city in the world."""

    url = "https://timezones.live/api/v1/time"

    response = requests.get(
        url,
        params={"city": city}
    )

    response.raise_for_status()

    data = response.json()["data"]

    return (
        f"Current time in {data['city']}, "
        f"{data['country']} is {data['currentTime']} "
        f"on {data['currentDate']}."
    )


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""

    return a + b


llm = ChatGroq(
    model=os.environ.get("GROQ_MODEL_OPEN"),
    temperature=0
)


tools = [
    get_current_time,
    add_numbers
]


agent = create_agent(
    model=llm,
    tools=tools
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What time is it in Tokyo?"
        }
    ]
})


print(result["messages"][-1].content)

for message in result["messages"]:
    print("=" * 70)
    print("MESSAGE TYPE:", type(message).__name__)
    print(message)

It’s currently **3:11 PM** in Tokyo (Mon, Sep 14).
MESSAGE TYPE: HumanMessage
content='What time is it in Tokyo?' additional_kwargs={} response_metadata={} id='596509fa-ff81-4b36-b4e6-679bf8e61546'
MESSAGE TYPE: AIMessage
content='' additional_kwargs={'reasoning_content': 'We need to call get_current_time with city "Tokyo".', 'tool_calls': [{'id': 'fc_6679bfcb-241d-4e01-a5ee-7c6cd8d4b44b', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_current_time'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 157, 'total_tokens': 194, 'completion_time': 0.03921609, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.013847194, 'prompt_tokens_details': None, 'queue_time': 0.319740716, 'total_time': 0.053063284}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09e8a-ea52-79

In [2]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 125 + 375?"
        }
    ]
})

print(result["messages"][-1].content)

for message in result["messages"]:
    print("=" * 70)
    print(type(message).__name__)
    print(message)

125 + 375 = **500**
HumanMessage
content='What is 125 + 375?' additional_kwargs={} response_metadata={} id='99051009-da76-482b-8d80-9521b42aa98c'
AIMessage
content='' additional_kwargs={'reasoning_content': "We need to add 125 + 375. That's 500. We can use add_numbers function.", 'tool_calls': [{'id': 'fc_425e2263-4081-414f-8907-420a5e373b18', 'function': {'arguments': '{"a":125,"b":375}', 'name': 'add_numbers'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 158, 'total_tokens': 207, 'completion_time': 0.050795955, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.009056512, 'prompt_tokens_details': None, 'queue_time': 0.333073868, 'total_time': 0.059852467}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66f3850a1a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09e8a-ff32-7860-a7f6-eb58309a4424-0' tool_calls=[{'name': 'add_